# 🧠 AI 초록 감별기 — 노트북 데모

학습된 모델(`best_koelectra_model.pt`, `lgb_models.pkl`, `pca_model.pkl`)을 불러와,
**논문 초록 텍스트를 직접 입력하고 AI 생성 여부를 실시간으로 감별**합니다.

---
### 📋 실행 순서
1. **Cell 1** — 라이브러리 설치 및 임포트
2. **Cell 2** — 저장된 모델 로드
3. **Cell 3** — 감별 함수 정의
4. **Cell 4** — 🎛️ 인터랙티브 데모 UI 실행

> 💡 **팁**: Cell을 순서대로 실행(Shift+Enter)하고, 마지막 Cell에서 초록을 입력한 뒤 **'🔍 감별 시작'** 버튼을 클릭하세요.

In [1]:
# ─────────────────────────────────────────────────────────
# Cell 1: 라이브러리 설치 및 임포트
# ─────────────────────────────────────────────────────────
!pip install -q lightgbm sentence-transformers transformers torch joblib ipywidgets

import os
import numpy as np
import torch
import joblib
import warnings
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    GPT2LMHeadModel,
    PreTrainedTokenizerFast
)

warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ 환경 준비 완료 | 사용 디바이스: {device.upper()}')


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ 환경 준비 완료 | 사용 디바이스: CUDA


In [2]:
# ─────────────────────────────────────────────────────────
# Cell 2: 저장된 모델 로드
# 이 폴더(ML_abstract_detector)에서 실행해야 모델 파일을 찾습니다.
# ─────────────────────────────────────────────────────────

MODEL_DIR = os.path.dirname(os.path.abspath('demo_detector.ipynb'))

print('🔄 모델 로드 중... (처음 실행 시 모델 다운로드로 수 분 소요될 수 있습니다)')

# 1) KoGPT2 - PPL(구조적 무작위성) 계산용
print('  [1/4] KoGPT2 로드 중...')
GPT_MODEL_NAME = 'skt/kogpt2-base-v2'
gpt_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    GPT_MODEL_NAME,
    bos_token='</s>', eos_token='</s>',
    unk_token='<unk>', pad_token='<pad>', mask_token='<mask>'
)
gpt_model = GPT2LMHeadModel.from_pretrained(GPT_MODEL_NAME).to(device)
gpt_model.eval()

# 2) KR-SBERT - 시맨틱 임베딩용
print('  [2/4] KR-SBERT 로드 중...')
sbert_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')

# 3) PCA + LightGBM - 저장된 pkl 파일 복원
print('  [3/4] LightGBM & PCA 모델 로드 중...')
pca_model  = joblib.load(os.path.join(MODEL_DIR, 'pca_model.pkl'))
lgb_models = joblib.load(os.path.join(MODEL_DIR, 'lgb_models.pkl'))

# 4) KoELECTRA Fine-tuned - 저장된 .pt 가중치 복원
print('  [4/4] KoELECTRA Fine-tuned 모델 로드 중...')
ELECTRA_NAME = 'monologg/koelectra-base-v3-discriminator'
electra_tokenizer = AutoTokenizer.from_pretrained(ELECTRA_NAME)
electra_model = AutoModelForSequenceClassification.from_pretrained(ELECTRA_NAME, num_labels=2)

pt_path = os.path.join(MODEL_DIR, 'best_koelectra_model.pt')
if os.path.exists(pt_path):
    electra_model.load_state_dict(torch.load(pt_path, map_location=device))
    print('  ✅ best_koelectra_model.pt 가중치 적용 완료')
else:
    print('  ⚠️  best_koelectra_model.pt 파일을 찾을 수 없습니다. 사전학습 가중치만 사용합니다.')

electra_model.to(device)
electra_model.eval()

print('\n🎉 모든 모델 로드 완료!')

🔄 모델 로드 중... (처음 실행 시 모델 다운로드로 수 분 소요될 수 있습니다)
  [1/4] KoGPT2 로드 중...


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [2/4] KR-SBERT 로드 중...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [3/4] LightGBM & PCA 모델 로드 중...
  [4/4] KoELECTRA Fine-tuned 모델 로드 중...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your do

  ✅ best_koelectra_model.pt 가중치 적용 완료

🎉 모든 모델 로드 완료!


In [3]:
# ─────────────────────────────────────────────────────────
# Cell 3: 감별 핵심 함수 정의
# ─────────────────────────────────────────────────────────

def calc_ppl(text: str) -> float:
    """KoGPT2로 텍스트의 Perplexity(구조적 무작위성)를 계산합니다."""
    if not isinstance(text, str) or not text.strip():
        return 50.0
    encodings = gpt_tokenizer(text, return_tensors='pt').to(device)
    max_length = gpt_model.config.n_positions
    stride = 512
    nlls = []
    seq_len = encodings.input_ids.size(1)
    end_loc = 0
    for i in range(0, seq_len, stride):
        begin_loc = max(i + stride - max_length, 0)
        end_loc   = min(i + stride, seq_len)
        trg_len   = end_loc - i
        input_ids   = encodings.input_ids[:, begin_loc:end_loc]
        target_ids  = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        with torch.no_grad():
            outputs = gpt_model(input_ids, labels=target_ids)
            nlls.append(outputs.loss * trg_len)
    if not nlls or end_loc == 0:
        return 50.0
    return torch.exp(torch.stack(nlls).sum() / end_loc).item()


def calc_ttr(text: str) -> float:
    """Type-Token Ratio(어휘 다양성)를 계산합니다."""
    tokens = text.split()
    return len(set(tokens)) / len(tokens) if tokens else 0.0


def predict(text: str) -> dict:
    """
    입력 텍스트에 대해 AI 생성 확률을 반환합니다.
    Returns:
        dict with keys: ppl, ttr, length, lgb_prob, electra_prob, final_prob
    """
    # ── 1. 수치 지표 계산 ──────────────────────
    ppl    = calc_ppl(text)
    ttr    = calc_ttr(text)
    length = len(text)

    # ── 2. SBERT 임베딩 + PCA ─────────────────
    emb     = sbert_model.encode([text])          # (1, 768)
    emb_pca = pca_model.transform(emb)[0]         # (16,)

    # ── 3. LightGBM 예측 ─────────────────────
    ppl_clipped = min(ppl, 300.0)
    X_tab = np.hstack([np.array([ppl_clipped, ttr, length]), emb_pca]).reshape(1, -1)
    lgb_prob = float(np.mean([m.predict_proba(X_tab)[:, 1][0] for m in lgb_models]))

    # ── 4. KoELECTRA 딥러닝 예측 ─────────────
    enc = electra_tokenizer(
        text, return_tensors='pt',
        padding=True, truncation=True, max_length=300
    ).to(device)
    with torch.no_grad():
        logits = electra_model(**enc).logits
    electra_prob = float(torch.softmax(logits, dim=1)[:, 1].cpu().item())

    # ── 5. 앙상블 (LGB 40% + ELECTRA 60%) ────
    final_prob = 0.4 * lgb_prob + 0.6 * electra_prob

    return {
        'ppl':          ppl,
        'ttr':          ttr,
        'length':       length,
        'lgb_prob':     lgb_prob,
        'electra_prob': electra_prob,
        'final_prob':   final_prob,
    }

print('✅ 감별 함수 정의 완료. Cell 4를 실행해 데모 UI를 시작하세요!')

✅ 감별 함수 정의 완료. Cell 4를 실행해 데모 UI를 시작하세요!


In [6]:
# ─────────────────────────────────────────────────────────
# Cell 4: 🎛️ 인터랙티브 감별 데모 UI
# ─────────────────────────────────────────────────────────

# ── UI 스타일 ──────────────────────────────────────────────
style_html = """
<style>
  .detector-card {
    background: #1e1e2e;
    border: 1px solid #313244;
    border-radius: 12px;
    padding: 20px 24px;
    font-family: 'Segoe UI', sans-serif;
    color: #cdd6f4;
    margin-top: 10px;
  }
  .detector-card h3 { margin: 0 0 14px; font-size: 1.1rem; color: #cba6f7; }
  .metric-row {
    display: flex; gap: 12px; margin-bottom: 14px; flex-wrap: wrap;
  }
  .metric-box {
    flex: 1; min-width: 120px;
    background: #313244;
    border-radius: 8px;
    padding: 10px 14px;
    text-align: center;
  }
  .metric-box .label { font-size: 0.72rem; color: #a6adc8; margin-bottom: 4px; }
  .metric-box .value { font-size: 1.3rem; font-weight: 700; color: #89dceb; }
  .metric-box .ref   { font-size: 0.65rem; color: #6c7086; margin-top: 3px; }
  .bar-wrap { margin: 14px 0; }
  .bar-label { display: flex; justify-content: space-between; font-size: 0.78rem; color: #a6adc8; margin-bottom: 4px; }
  .bar-bg {
    background: #313244; border-radius: 50px; height: 18px; overflow: hidden;
  }
  .bar-fill {
    height: 100%; border-radius: 50px;
    transition: width 0.5s ease;
  }
  .verdict-ai    { background: linear-gradient(90deg, #f38ba8, #fab387); }
  .verdict-human { background: linear-gradient(90deg, #a6e3a1, #94e2d5); }
  .verdict-box {
    border-radius: 10px; padding: 14px 18px;
    margin-top: 14px; font-size: 1rem; font-weight: 600;
  }
  .verdict-box.ai    { background: #3b0f1a; border: 1px solid #f38ba8; color: #f38ba8; }
  .verdict-box.human { background: #0d2b1f; border: 1px solid #a6e3a1; color: #a6e3a1; }
  .prob-detail { font-size: 0.75rem; color: #a6adc8; margin-top: 8px; }
</style>
"""
display(HTML(style_html))

# ── 위젯 정의 ─────────────────────────────────────────────
text_area = widgets.Textarea(
    placeholder='📄 여기에 논문 초록을 붙여넣고 아래 버튼을 클릭하세요...',
    layout=widgets.Layout(width='100%', height='180px')
)

run_button = widgets.Button(
    description=' 🔍 감별 시작',
    button_style='',
    layout=widgets.Layout(width='160px', height='38px'),
    style={'button_color': '#cba6f7', 'font_weight': 'bold'}
)

clear_button = widgets.Button(
    description=' 🗑️ 초기화',
    button_style='',
    layout=widgets.Layout(width='120px', height='38px'),
    style={'button_color': '#45475a'}
)

output_area = widgets.Output()
status_bar  = widgets.HTML(value='')

# ── 버튼 콜백 ─────────────────────────────────────────────
def on_run_clicked(b):
    text = text_area.value.strip()
    if not text:
        with output_area:
            clear_output()
            display(HTML('<p style="color:#f38ba8">⚠️ 초록 텍스트를 입력해주세요.</p>'))
        return

    status_bar.value = '<span style="color:#f9e2af">⏳ 분석 중... PPL 계산은 수십 초 걸릴 수 있습니다.</span>'
    run_button.disabled = True

    try:
        result = predict(text)

        p         = result['final_prob']
        pct       = p * 100
        is_ai     = p >= 0.5
        bar_cls   = 'verdict-ai' if is_ai else 'verdict-human'
        box_cls   = 'ai'         if is_ai else 'human'
        verdict   = '🚨 AI 생성물로 판정됩니다!' if is_ai else '✅ 인간 작성물로 판정됩니다!'
        hint      = ('낮은 구조적 무작위성(PPL)과 높은 어휘 다양성(TTR)이 감지되었습니다. '
                     'AI는 단어 중복을 기계적으로 회피하여 TTR이 높게 나타나는 특성이 있습니다.'
                     if is_ai else
                     '높은 구조적 무작위성(PPL)과 인간 특유의 자연스러운 단어 반복 패턴(낮은 TTR)이 확인되었습니다.')

        html = f"""
        <div class="detector-card">
          <h3>📊 분석 리포트</h3>
          <div class="metric-row">
            <div class="metric-box">
              <div class="label">구조적 무작위성 (PPL)</div>
              <div class="value">{result['ppl']:.1f}</div>
              <div class="ref">Human avg 73.1 (높음) / AI avg 54.3 (낮음)</div>
            </div>
            <div class="metric-box">
              <div class="label">어휘 다양성 (TTR)</div>
              <div class="value">{result['ttr']:.4f}</div>
              <div class="ref">Human avg 0.86 (낮음) / AI avg 0.90 (높음)</div>
            </div>
            <div class="metric-box">
              <div class="label">초록 글자 수</div>
              <div class="value">{result['length']:,}</div>
              <div class="ref">문자 수 기준</div>
            </div>
          </div>

          <div class="bar-wrap">
            <div class="bar-label">
              <span>AI 생성 확률</span>
              <span><b>{pct:.1f}%</b></span>
            </div>
            <div class="bar-bg">
              <div class="bar-fill {bar_cls}" style="width:{pct:.1f}%"></div>
            </div>
          </div>

          <div class="verdict-box {box_cls}">
            {verdict}<br>
            <span style="font-weight:400;font-size:0.85rem">{hint}</span>
          </div>

          <div class="prob-detail">
            LightGBM 예측: {result['lgb_prob']*100:.1f}% &nbsp;|&nbsp;
            KoELECTRA 예측: {result['electra_prob']*100:.1f}% &nbsp;|&nbsp;
            앙상블 (LGB×0.4 + ELECTRA×0.6): <b>{pct:.1f}%</b>
          </div>
        </div>
        """

        with output_area:
            clear_output()
            display(HTML(html))

        status_bar.value = '<span style="color:#a6e3a1">✅ 분석 완료</span>'

    except Exception as e:
        with output_area:
            clear_output()
            display(HTML(f'<p style="color:#f38ba8">❌ 오류 발생: {e}</p>'))
        status_bar.value = '<span style="color:#f38ba8">오류 발생</span>'
    finally:
        run_button.disabled = False


def on_clear_clicked(b):
    text_area.value = ''
    with output_area:
        clear_output()
    status_bar.value = ''


run_button.on_click(on_run_clicked)
clear_button.on_click(on_clear_clicked)

# ── 레이아웃 렌더링 ───────────────────────────────────────
display(HTML('<h3 style="font-family:Segoe UI,sans-serif;color:#cba6f7">🧠 AI 초록 감별기 — 인터랙티브 데모</h3>'))
display(HTML('<p style="font-family:Segoe UI,sans-serif;color:#a6adc8;margin-top:-8px">논문 초록을 붙여넣고 감별 버튼을 클릭하세요.</p>'))
display(text_area)
display(widgets.HBox([run_button, clear_button]))
display(status_bar)
display(output_area)


Textarea(value='', layout=Layout(height='180px', width='100%'), placeholder='📄 여기에 논문 초록을 붙여넣고 아래 버튼을 클릭하세요...…

HTML(value='')

Output()